Packages import

In [1]:
import requests
import os
import pandas as pd
import yaml
from requests.auth import HTTPBasicAuth
from bs4 import BeautifulSoup

Apollo Scraper

In [ ]:
with open("config.yaml", "r", encoding="UTF-8") as yf:
    config = yaml.load(yf)
username = config['credentials']['username']
password = config['credentials']['password']
credentials = HTTPBasicAuth(username, password)

In [ ]:
group_id = input("Enter your group id : ")
url = f"https://planzajec.uek.krakow.pl/index.php?typ=G&id=252671&okres=1"
response = requests.get(url, auth=credentials)
response.encoding = "UTF-8"
print(response.status_code)

In [ ]:
page_dom = BeautifulSoup(response.text, 'html.parser')

In [ ]:
group = page_dom.select_one("div.grupa").get_text(strip=True)
print(group)

In [ ]:
classes_tag = page_dom.select_one("table")
with open("temp.html", "w", encoding="UTF-8") as hf:
    hf.write(str(classes_tag))

classes = pd.read_html("temp.html")[0]
os.remove("temp.html")

In [ ]:
classes = classes.loc[classes['Typ'].isin(["ćwiczenia", "wykład", "egzamin"])]

In [ ]:
classes[['Date', 'Start time', 'hyphen', 'End time', 'Duration']] = classes['Dzień', 'godzina'].str.split(' ', expand=True)

In [ ]:
classes['Duration'] = classes['Duration'].map(lambda x: x.split('(')[1].split('g')[0])

In [ ]:
classes['hyphen'] = classes.drop(['Dzień, godzina', 'hyphen'], axis=1)

In [ ]:
classes['Sala'] = classes['Sala'].str.replace(
    r"(lab\.).*",
    r"\1",
    regex=True
)

In [ ]:
if not os.path.exists("schedules"):
    os.mkdir("schedules")

In [ ]:
classes.to_csv(f"schedules/{group}.csv")